# DDRI 시간·요일 원형 인코딩 비교 노트북

이 노트북은 `hour`, `weekday`를 그대로 쓰는 기준선과,
`hour_sin`, `hour_cos`, `weekday_sin`, `weekday_cos`를 추가한 조건을 직접 비교한다.

원칙:
- 스크립트 호출 래퍼가 아니라 노트북 안에서 단계별로 직접 실행한다.
- 임포트 셀은 항상 별도로 둔다.
- 기준선과 원형 인코딩 조건의 결과를 같은 정책으로 비교한다.


## 1. 임포트

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

import pandas as pd


## 1-1. 커널과 의존성 확인

In [ ]:
print('python executable:', sys.executable)
for module_name in ['lightgbm', 'catboost', 'sklearn', 'pandas', 'numpy']:
    try:
        __import__(module_name)
        print(module_name, 'OK')
    except Exception as exc:
        print(module_name, 'MISSING', type(exc).__name__, str(exc))


## 2. 모듈과 대상 데이터셋 선택

In [ ]:
ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works' / '06_prediction_training'
MODEL_SCRIPT = WORK_DIR / '05_scripts' / '04_ddri_prediction_train_eval.py'
FEATURE_SCRIPT = WORK_DIR / '05_scripts' / '03_ddri_prediction_modeling_feature_builder.py'

dataset_name = 'rep15'  # 'rep15' 또는 'full161'
target_col = 'bike_change_deseasonalized'  # 또는 'bike_change_raw'

feature_spec = importlib.util.spec_from_file_location('feature_builder_cyclic', FEATURE_SCRIPT)
feature_builder = importlib.util.module_from_spec(feature_spec)
sys.modules[feature_spec.name] = feature_builder
feature_spec.loader.exec_module(feature_builder)

train_spec = importlib.util.spec_from_file_location('train_eval_cyclic', MODEL_SCRIPT)
train_eval = importlib.util.module_from_spec(train_spec)
sys.modules[train_spec.name] = train_eval
train_spec.loader.exec_module(train_eval)

modeling_spec = next(spec for spec in feature_builder.SPECS if spec.name == dataset_name)
run_spec = train_eval.SPECS[dataset_name]
modeling_spec, run_spec


## 3. canonical_data 입력 확인

In [ ]:
canonical_train_path = modeling_spec.canonical_dir / 'ddri_prediction_canonical_train_2023_2024.csv'
canonical_test_path = modeling_spec.canonical_dir / 'ddri_prediction_canonical_test_2025.csv'

pd.DataFrame([
    {'name': 'canonical_train', 'path': str(canonical_train_path), 'exists': canonical_train_path.exists(), 'size_bytes': canonical_train_path.stat().st_size if canonical_train_path.exists() else None},
    {'name': 'canonical_test', 'path': str(canonical_test_path), 'exists': canonical_test_path.exists(), 'size_bytes': canonical_test_path.stat().st_size if canonical_test_path.exists() else None},
])


## 4. 정본 로드 및 연도 분할 확인

In [ ]:
canonical_train = pd.read_csv(canonical_train_path)
canonical_test = pd.read_csv(canonical_test_path)
canonical_train['date'] = pd.to_datetime(canonical_train['date'])
canonical_test['date'] = pd.to_datetime(canonical_test['date'])

train_2023 = canonical_train[canonical_train['date'].dt.year == 2023].copy()
valid_2024 = canonical_train[canonical_train['date'].dt.year == 2024].copy()
test_2025 = canonical_test.copy()

pd.DataFrame([
    {'frame': 'train_2023', 'rows': len(train_2023), 'stations': train_2023['station_id'].nunique()},
    {'frame': 'valid_2024', 'rows': len(valid_2024), 'stations': valid_2024['station_id'].nunique()},
    {'frame': 'test_2025', 'rows': len(test_2025), 'stations': test_2025['station_id'].nunique()},
])


## 5. 기준선 전처리와 원형 인코딩 전처리 생성

In [ ]:
train_base = feature_builder.build_modeling_frame(train_2023, add_cyclic=False)
valid_base = feature_builder.build_modeling_frame(valid_2024, add_cyclic=False)
test_base = feature_builder.build_modeling_frame(test_2025, add_cyclic=False)

train_cyclic = feature_builder.build_modeling_frame(train_2023, add_cyclic=True)
valid_cyclic = feature_builder.build_modeling_frame(valid_2024, add_cyclic=True)
test_cyclic = feature_builder.build_modeling_frame(test_2025, add_cyclic=True)

pd.DataFrame([
    {'variant': 'baseline', 'train_cols': len(train_base.columns), 'contains_hour_sin': 'hour_sin' in train_base.columns, 'contains_weekday_sin': 'weekday_sin' in train_base.columns},
    {'variant': 'cyclic', 'train_cols': len(train_cyclic.columns), 'contains_hour_sin': 'hour_sin' in train_cyclic.columns, 'contains_weekday_sin': 'weekday_sin' in train_cyclic.columns},
])


## 6. 추가된 원형 인코딩 피처 확인

In [ ]:
train_cyclic[['hour', 'weekday', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos']].head(12)


## 7. 공통 학습 함수 정의

In [ ]:
def run_cycle(train_model: pd.DataFrame, valid_model: pd.DataFrame, test_model: pd.DataFrame, target_col: str) -> tuple[pd.DataFrame, pd.DataFrame, list[str], str]:
    train_fit = train_model[train_model[target_col].notna()].copy()
    valid_fit = valid_model[valid_model[target_col].notna()].copy()
    test_fit = test_model[test_model[target_col].notna()].copy()

    feature_cols = train_eval.build_feature_cols(train_fit)
    X_train = train_fit[feature_cols]
    y_train = train_fit[target_col]
    X_valid = valid_fit[feature_cols]
    y_valid = valid_fit[target_col]
    X_test = test_fit[feature_cols]
    y_test = test_fit[target_col]

    selection_rows = []
    for model_name, model in train_eval.candidate_models():
        model.fit(X_train, y_train)
        pred_valid = model.predict(X_valid)
        selection_rows.append({'model': model_name, **train_eval.metrics(y_valid, pred_valid)})

    selection_df = pd.DataFrame(selection_rows).sort_values(['rmse', 'mae', 'r2'], ascending=[True, True, False]).reset_index(drop=True)
    best_model_name = selection_df.iloc[0]['model']

    final_X = pd.concat([X_train, X_valid], ignore_index=True)
    final_y = pd.concat([y_train, y_valid], ignore_index=True)
    final_model = dict(train_eval.candidate_models())[best_model_name]
    final_model.fit(final_X, final_y)

    test_pred = final_model.predict(X_test)
    test_metrics_df = pd.DataFrame([{'model': best_model_name, **train_eval.metrics(y_test, test_pred)}])
    return selection_df, test_metrics_df, feature_cols, best_model_name


## 8. 기준선 실행

In [ ]:
selection_base, test_base_metrics, feature_cols_base, best_base = run_cycle(train_base, valid_base, test_base, target_col)
selection_base


## 9. 원형 인코딩 조건 실행

In [ ]:
selection_cyclic, test_cyclic_metrics, feature_cols_cyclic, best_cyclic = run_cycle(train_cyclic, valid_cyclic, test_cyclic, target_col)
selection_cyclic


## 10. 테스트 결과 비교

In [ ]:
compare_df = pd.concat([
    pd.DataFrame([{'variant': 'baseline', **test_base_metrics.iloc[0].to_dict()}]),
    pd.DataFrame([{'variant': 'cyclic', **test_cyclic_metrics.iloc[0].to_dict()}]),
], ignore_index=True)
compare_df


## 11. 추가된 피처 수 비교

In [ ]:
pd.DataFrame([
    {'variant': 'baseline', 'feature_count': len(feature_cols_base)},
    {'variant': 'cyclic', 'feature_count': len(feature_cols_cyclic), 'added_features': ', '.join(sorted(set(feature_cols_cyclic) - set(feature_cols_base)))},
])


## 12. 결과 저장

In [ ]:
output_dir_map = {
    'rep15': ROOT / '3조 공유폴더' / '대표대여소_예측데이터_15개' / 'cyclic_time_feature_outputs',
    'full161': ROOT / '3조 공유폴더' / '군집별 데이터_전체 스테이션' / 'cyclic_time_feature_outputs',
}
output_dir = output_dir_map[dataset_name]
training_dir = output_dir / 'training_runs'
training_dir.mkdir(parents=True, exist_ok=True)

selection_compare = pd.concat([
    selection_base.assign(variant='baseline'),
    selection_cyclic.assign(variant='cyclic'),
], ignore_index=True)

test_compare = compare_df.copy()

selection_path = training_dir / f'ddri_{dataset_name}_{target_col}_selection_metrics_cyclic_compare.csv'
test_path = training_dir / f'ddri_{dataset_name}_{target_col}_test_metrics_cyclic_compare.csv'
meta_path = training_dir / f'ddri_{dataset_name}_{target_col}_cyclic_compare_meta.json'

selection_compare.to_csv(selection_path, index=False, encoding='utf-8-sig')
test_compare.to_csv(test_path, index=False, encoding='utf-8-sig')
meta_path.write_text(json.dumps({
    'dataset': dataset_name,
    'target': target_col,
    'baseline_best_model': best_base,
    'cyclic_best_model': best_cyclic,
    'baseline_feature_count': len(feature_cols_base),
    'cyclic_feature_count': len(feature_cols_cyclic),
    'added_cyclic_features': sorted(set(feature_cols_cyclic) - set(feature_cols_base)),
    'selection_output': str(selection_path),
    'test_output': str(test_path),
}, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([
    {'name': 'selection_compare', 'path': str(selection_path)},
    {'name': 'test_compare', 'path': str(test_path)},
    {'name': 'meta', 'path': str(meta_path)},
])
